# Post-fit pulls in physical parameters — asimov_fit_results

This compact notebook transforms each fit's **joint MCMC chain** from standardized fit coordinates to the physical $M_A$ or complete $z$-expansion coefficient vector. For correlated priors it applies the inverse PCA map. The MINERvA kmax=6 uniform fit uses the same PCA spline directions with flat penalties; its corner plot is posterior-only because no informative prior enters the fit. Dependent coefficients preserve $F_A(0)$ and the four sum rules.

In [ ]:
from pathlib import Path
import sys

repo = Path.cwd().resolve()
while repo.name != 'axial_mass' and repo != repo.parent:
    repo = repo.parent
helper_dir = repo / 'ma_zexp' / 'python' / 'scripts'
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

from postfit_physical_parameters import (
    FIGURE_ROOT, plot_distribution_overlay, plot_ma_posterior_overlay, run_suite,
)

## Run every measurement

Set `BURN_IN` or `THIN` if needed. Each measurement produces one table and one two-row figure: transformed marginal distributions above, prior/post-fit intervals below.

In [ ]:
BURN_IN = 0
THIN = 1
N_PRIOR_SAMPLES = 50_000
SHOW_PRIOR_IN_CORNER = True
SAVE_FIGURES = True
SAVE_DPI = 600

results = run_suite(
    'asimov_fit_results', BURN_IN, THIN, N_PRIOR_SAMPLES,
    show_prior_in_corner=SHOW_PRIOR_IN_CORNER,
    save_figures=SAVE_FIGURES, save_dpi=SAVE_DPI,
)

## Custom comparable overlay corner

Choose any available z-expansion posteriors and reference priors below. The large contour-only panel compares physical $(a_1,a_2)$ distributions. It verifies that every selection uses the same $t_0$ and $t_\mathrm{cut}$ basis before plotting. Solid contours are posteriors and dashed contours are priors.

In [ ]:
# Each entry is (source key, 'prior' or 'posterior').
# Priors may use any loaded fit key, plus the standalone references
# deuterium, deuterium_k6, minerva_k6, lqcd_k6, and minerva_lqcd_k6.
# Posteriors use successfully loaded fit names in results.
OVERLAY_DISTRIBUTIONS = [
    ('minerva_k6_uniform', 'posterior'),
    ('deuterium_k6', 'prior'),
    ('minerva_k6', 'prior'),
    ('lqcd_k6', 'prior'),
]

overlay_corner = plot_distribution_overlay(
    results, OVERLAY_DISTRIBUTIONS, figsize=(7.0, 6.2),
)
overlay_dir = FIGURE_ROOT / 'asimov_fit_results' / 'comparison_overlays'
overlay_dir.mkdir(parents=True, exist_ok=True)
for extension in ('pdf', 'png'):
    overlay_corner.savefig(
        overlay_dir / f'coefficient_overlay.{extension}', dpi=600,
        bbox_inches='tight', pad_inches=.03, facecolor='white',
    )
overlay_corner

## $M_A$ fit with and without its pull penalty

This corner overlays the joint posteriors for physical $M_A$, `NormCCMEC`, and `RPA_CCQE`. Both fits omit `AxFFCCQEshape`; the only prior difference is that `ma_uniform` removes the Gaussian pull penalty from $M_A$.

In [ ]:
MA_OVERLAY_FITS = ('ma_no_axff', 'ma_uniform')
MA_OVERLAY_LABELS = {
    'ma_no_axff': r'$M_A$ Gaussian prior',
    'ma_uniform': r'$M_A$ uniform',
}

ma_overlay_corner = plot_ma_posterior_overlay(
    results, MA_OVERLAY_FITS, labels=MA_OVERLAY_LABELS,
    figsize=(7.2, 7.2),
)
ma_overlay_dir = FIGURE_ROOT / 'asimov_fit_results' / 'comparison_overlays'
ma_overlay_dir.mkdir(parents=True, exist_ok=True)
for extension in ('pdf', 'png'):
    ma_overlay_corner.savefig(
        ma_overlay_dir / f'ma_overlay.{extension}', dpi=600,
        bbox_inches='tight', pad_inches=.03, facecolor='white',
    )
ma_overlay_corner